
# Example 3b — Nonlinear cylinder variance collapse with a variable number of particles

This notebook keeps the **genuine nonlinear-cylinder / Algorithm 2 \(h\)-transform** from the paper and makes the
particle count easy to change.

What changed relative to the earlier example:

- the initial square is now populated by **\(n\) equally spaced particles on the square perimeter**
- by default the masses are **equal**, \(s_i = 1/n\)
- the top configuration cell exposes `n_particles`, so you can change the count and rerun from there
- for equal masses, the notebook uses a **fast exact vectorization** of the same Algorithm 2 update

Why the fast path is still the same \(h\)-transform:
- in Algorithm 2, at a fixed time step \(m\) and quadrature node \(q\), each particle uses
  \[
  P_{\tau/s_i}\!\left(e^{i\,s_i\,\eta_q\,\phi}\right)(x_i),
  \qquad
  \nabla P_{\tau/s_i}\!\left(e^{i\,s_i\,\eta_q\,\phi}\right)(x_i)
  \]
- when all masses are equal, \(s_i \equiv 1/n\), the **weight** \(s_i\eta_q\) and the **heat time** \(\tau/s_i\) are the same for every particle
- so we can evaluate those Fourier sums for **all particle locations at once** and then apply the same prefix/suffix products as in the reference code

So the fast routine below is **not a different model** and **not a different conditioning rule**.
It is the same quadrature + heat-semigroup + Euler–Maruyama scheme, only vectorized for the equal-mass case.

A second small change is that the horizon is scaled like
\[
T_n = \frac{c}{n}.
\]
For equal masses, this keeps both the Brownian noise scale \(\sqrt{2T_n/s_i}\) and the heat time \(T_n/s_i\) at an \(O(1)\) level as \(n\) changes, which makes the example much more comparable across particle counts.


In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sys
from pathlib import Path
from numpy.polynomial.hermite import hermgauss


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.support import configure_plotly
from wasserstein_conditioning_algorithms import (
    ParticleSimulation,
    TorusFourierHeatSemigroup,
    _as_float_array,
    _build_time_grid,
    _get_rng,
    _prefix_suffix_products,
    _validate_positions,
    _validate_probability_vector,
    shortest_periodic_displacement,
    simulate_nonlinear_cylinder_quadrature_em,
    wrap_torus,
)

configure_plotly()
np.set_printoptions(precision=3, suppress=True)


In [ ]:
from notebooks.support import (
    center_trace,
    circle_trace,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        particle_name="particles",
        mass_format=".4f",
        time_formatter=lambda t, h: f"time = {t:.4f}",
        slider_label_formatter=lambda t, h: f"{t:.4f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=110,
    )


In [ ]:

def cosine_blob_observable(center=(0.5, 0.5)):
    center = np.asarray(center, dtype=float)

    def phi(points):
        pts = np.asarray(points, dtype=float)
        return 0.5 * (
            np.cos(2.0 * np.pi * (pts[..., 0] - center[0]))
            + np.cos(2.0 * np.pi * (pts[..., 1] - center[1]))
        )

    return phi

def gauss_hermite_cylinder_quadrature(lambda_, order):
    nodes, weights = hermgauss(order)
    eta = np.sqrt(2.0 * lambda_) * nodes[:, None]
    quad_weights = weights / np.sqrt(np.pi)
    return eta, quad_weights

def weighted_observable_score(positions, masses, observable):
    return np.array([np.sum(masses * observable(pos)) for pos in positions], dtype=float)

def weighted_periodic_rms_radius(positions, masses, center):
    center = np.asarray(center, dtype=float)
    displacements = shortest_periodic_displacement(positions, center[None, None, :])
    squared_radii = np.sum(displacements ** 2, axis=-1)
    return np.sqrt(squared_radii @ masses)

def simulate_free_diffusion(masses, initial_positions, horizon, step_size, seed):
    rng = np.random.default_rng(seed)
    masses = np.asarray(masses, dtype=float)
    positions0 = np.mod(np.asarray(initial_positions, dtype=float), 1.0)

    m_steps = int(round(horizon / step_size))
    times = np.linspace(0.0, horizon, m_steps + 1, dtype=float)
    positions = np.empty((m_steps + 1, len(masses), positions0.shape[1]), dtype=float)
    positions[0] = positions0

    state = positions0.copy()
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(m_steps):
        state = np.mod(state + noise_scale * rng.normal(size=state.shape), 1.0)
        positions[m + 1] = state

    return times, positions

def make_square_perimeter_configuration(n, center=(0.5, 0.5), half_width=0.35, phase=0.0):
    cx, cy = np.asarray(center, dtype=float)
    perimeter = 8.0 * half_width
    arclength = ((np.arange(n, dtype=float) / n) + phase) % 1.0
    arclength = arclength * perimeter

    pts = np.empty((n, 2), dtype=float)
    for i, value in enumerate(arclength):
        if value < 2.0 * half_width:
            pts[i] = [cx - half_width + value, cy - half_width]
        elif value < 4.0 * half_width:
            pts[i] = [cx + half_width, cy - half_width + (value - 2.0 * half_width)]
        elif value < 6.0 * half_width:
            pts[i] = [cx + half_width - (value - 4.0 * half_width), cy + half_width]
        else:
            pts[i] = [cx - half_width, cy + half_width - (value - 6.0 * half_width)]
    return np.mod(pts, 1.0)

def square_outline_from_center(center=(0.5, 0.5), half_width=0.35):
    cx, cy = np.asarray(center, dtype=float)
    return np.array([
        [cx - half_width, cy - half_width],
        [cx + half_width, cy - half_width],
        [cx + half_width, cy + half_width],
        [cx - half_width, cy + half_width],
    ], dtype=float)



### Exact equal-mass fast path for Algorithm 2

The next cell is just a vectorized implementation of the **same** nonlinear-cylinder update used in
`simulate_nonlinear_cylinder_quadrature_em(...)` when all masses are equal.

If you later replace the equal masses by a non-uniform mass vector, the wrapper falls back to the
reference implementation from `wasserstein_conditioning_algorithms.py`.


In [ ]:

def evaluate_many_equal_mass(solver, weight, time, points):
    coeffs = solver._coefficients(weight)
    decay = np.exp(-4.0 * np.pi ** 2 * solver.k_sq_norm * time)
    base = coeffs * decay

    x = wrap_torus(np.asarray(points, dtype=float))
    phase = np.exp(2j * np.pi * (solver.k_vectors @ x.T))
    terms = base[:, None] * phase

    values = np.sum(terms, axis=0)
    gradients = (terms.T @ (2j * np.pi * solver.k_vectors)).astype(np.complex128)
    return values, gradients

def simulate_nonlinear_cylinder_quadrature_em_equal_mass_fast(
    masses,
    observables,
    target_vector,
    lambda_,
    horizon,
    step_size,
    initial_positions,
    quadrature_nodes,
    quadrature_weights,
    *,
    grid_shape=32,
    rng=None,
    store_drifts=True,
):
    if lambda_ <= 0.0 or not np.isfinite(lambda_):
        raise ValueError("lambda_ must be positive and finite")

    s = _validate_probability_vector(masses, name="masses")
    if not np.allclose(s, s[0], atol=1e-14, rtol=1e-14):
        raise ValueError("equal-mass fast path requires all masses to be equal")

    n = len(s)
    x0 = wrap_torus(_validate_positions(initial_positions, n=n, name="initial_positions"))
    _, d = x0.shape
    s0 = float(s[0])

    a = _as_float_array(target_vector, name="target_vector").reshape(-1)
    eta = _as_float_array(quadrature_nodes, name="quadrature_nodes")
    if eta.ndim == 1:
        eta = eta[:, None]
    if eta.shape[1] != len(a):
        raise ValueError(f"quadrature_nodes must have shape (Q, {len(a)}) or (Q,)")

    weights = _as_float_array(quadrature_weights, name="quadrature_weights").reshape(-1)
    if len(weights) != len(eta):
        raise ValueError("quadrature_nodes and quadrature_weights must have matching lengths")

    solver = TorusFourierHeatSemigroup(observables, dimension=d, grid_shape=grid_shape)
    rng = _get_rng(rng)
    m_steps, times = _build_time_grid(horizon, step_size)

    positions = np.empty((m_steps + 1, n, d), dtype=np.float64)
    positions[0] = x0
    drift_history = np.empty((m_steps, n, d), dtype=np.float64) if store_drifts else None
    noise_scale = np.sqrt(2.0 * step_size / s)[:, None]

    q_count = len(weights)

    for m in range(m_steps):
        tau = horizon - times[m]
        x = positions[m]
        common_time = tau / s0

        h = np.empty((q_count, n), dtype=np.complex128)
        gamma = np.empty((q_count, n, d), dtype=np.complex128)

        for q in range(q_count):
            values, grads = evaluate_many_equal_mass(solver, s0 * eta[q], common_time, x)
            h[q] = values
            gamma[q] = grads

        u_terms = np.empty(q_count, dtype=np.complex128)
        g_terms = np.empty((q_count, n, d), dtype=np.complex128)

        for q in range(q_count):
            prefactor = weights[q] * np.exp(-1j * float(np.dot(eta[q], a)))
            prefix, suffix = _prefix_suffix_products(h[q])
            u_terms[q] = prefactor * prefix[-1]
            g_terms[q] = prefactor * gamma[q] * prefix[:-1, None] * suffix[1:, None]

        u_value = float(np.real(np.sum(u_terms)))
        if not np.isfinite(u_value) or u_value <= 1e-14:
            raise FloatingPointError(
                "quadrature approximation produced a near-zero denominator; "
                "increase quadrature accuracy or adjust the Fourier grid"
            )

        g_sum = np.sum(g_terms, axis=0)
        drift = (2.0 / s0) * np.real(g_sum / u_value)

        positions[m + 1] = wrap_torus(
            x + drift * step_size + noise_scale * rng.normal(size=(n, d))
        )
        if drift_history is not None:
            drift_history[m] = drift

    return ParticleSimulation(times=times, positions=positions, masses=s, drifts=drift_history)

def simulate_algorithm2_exact(
    masses,
    observables,
    target_vector,
    lambda_,
    horizon,
    step_size,
    initial_positions,
    quadrature_nodes,
    quadrature_weights,
    *,
    grid_shape=32,
    rng=None,
    store_drifts=True,
):
    masses = np.asarray(masses, dtype=float)
    if np.allclose(masses, masses[0], atol=1e-14, rtol=1e-14):
        return simulate_nonlinear_cylinder_quadrature_em_equal_mass_fast(
            masses=masses,
            observables=observables,
            target_vector=target_vector,
            lambda_=lambda_,
            horizon=horizon,
            step_size=step_size,
            initial_positions=initial_positions,
            quadrature_nodes=quadrature_nodes,
            quadrature_weights=quadrature_weights,
            grid_shape=grid_shape,
            rng=rng,
            store_drifts=store_drifts,
        )
    return simulate_nonlinear_cylinder_quadrature_em(
        masses=masses,
        observables=observables,
        target_vector=target_vector,
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        quadrature_nodes=quadrature_nodes,
        quadrature_weights=quadrature_weights,
        grid_shape=grid_shape,
        rng=rng,
        store_drifts=store_drifts,
    )



### Configuration

Change `n_particles` here and rerun the notebook from this cell downward.

The default uses equal masses and places the particles on the perimeter of a large square, so the
variance-collapse effect stays easy to see.


In [ ]:

# --- main configuration ---
n_particles = 12
seed = 0

# geometry / target
target_center = np.array([0.50, 0.50], dtype=float)
square_half_width = 0.35
compact_guide_radius = 0.18

# cylinder target
target_score = 0.80
lambda_ = 20.0

# discretization
n_steps = 100
base_horizon_per_particle = 0.08
quadrature_order = 21
grid_shape = 40

# diagnostics
robustness_seeds = list(range(5))
benchmark_counts = [4, 8, 12, 16, 24, 32]

# --- derived quantities ---
masses = np.ones(n_particles, dtype=float) / n_particles
initial_positions = make_square_perimeter_configuration(
    n_particles,
    center=target_center,
    half_width=square_half_width,
)
square_outline = square_outline_from_center(target_center, square_half_width)
observable = cosine_blob_observable(target_center)

horizon = base_horizon_per_particle / n_particles
step_size = horizon / n_steps
quadrature_nodes, quadrature_weights = gauss_hermite_cylinder_quadrature(lambda_, quadrature_order)

def run_conditioned_simulation(seed, *, store_drifts=True):
    rng = np.random.default_rng(seed)
    return simulate_algorithm2_exact(
        masses=masses,
        observables=[observable],
        target_vector=np.array([target_score], dtype=float),
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        quadrature_nodes=quadrature_nodes,
        quadrature_weights=quadrature_weights,
        grid_shape=grid_shape,
        rng=rng,
        store_drifts=store_drifts,
    )

initial_score = float(np.sum(masses * observable(initial_positions)))
initial_rms_radius = float(
    weighted_periodic_rms_radius(initial_positions[None, :, :], masses, target_center)[0]
)

marker_size = float(np.clip(38.0 - 0.55 * n_particles, 9.0, 28.0))

print(f"n_particles = {n_particles}")
print(f"horizon = {horizon:.6f}  (scaled like 0.08 / n)")
print(f"step_size = {step_size:.6f}")
print(f"quadrature_order = {quadrature_order}, grid_shape = {grid_shape}")
print(f"initial weighted cosine score = {initial_score:.6f}")
print(f"initial weighted periodic RMS radius = {initial_rms_radius:.6f}")
print(f"target score a = {target_score:.3f}")


In [ ]:

t0 = time.perf_counter()
sim = run_conditioned_simulation(seed, store_drifts=True)
conditioned_runtime = time.perf_counter() - t0

free_times, free_positions = simulate_free_diffusion(masses, initial_positions, horizon, step_size, seed)

score = weighted_observable_score(sim.positions, sim.masses, observable)
rms_radius = weighted_periodic_rms_radius(sim.positions, sim.masses, target_center)
free_rms_radius = weighted_periodic_rms_radius(free_positions, masses, target_center)

print("positions array shape:", sim.positions.shape)
print("conditioned runtime (seconds):", round(conditioned_runtime, 3))
print("final time:", float(sim.times[-1]))
print("final conditioned score:", float(score[-1]))
print("final conditioned RMS radius:", float(rms_radius[-1]))
print("final free-diffusion RMS radius:", float(free_rms_radius[-1]))
print("conditioned / free final-radius ratio:", float(rms_radius[-1] / free_rms_radius[-1]))


In [ ]:

static_traces = [
    line_trace(square_outline, name="initial square guide", color="rgba(30, 144, 255, 0.50)", dash="dash", close=True),
    circle_trace(target_center, radius=compact_guide_radius, name="compact guide (visual only)", color="rgba(220, 20, 60, 0.55)"),
    center_trace(target_center[None, :], name="target center", color="rgba(220, 20, 60, 0.8)", symbol="x", size=12, showlegend=False),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title=f"Exact Algorithm 2 variance collapse (n = {n_particles})",
    static_traces=static_traces,
    marker_size=marker_size,
)
fig.show()



### Free-diffusion baseline

Same initial condition, same seed, but no conditioning drift.


In [ ]:

free_fig = make_particle_animation(
    positions=free_positions,
    times=free_times,
    masses=masses,
    title=f"Free diffusion baseline (n = {n_particles}, same initial state and same seed)",
    static_traces=static_traces,
    marker_size=marker_size,
)
free_fig.show()


In [ ]:

score_fig = go.Figure()
score_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=score,
        mode="lines",
        name="conditioned weighted cosine score",
    )
)
score_fig.add_hline(
    y=target_score,
    line_dash="dash",
    annotation_text="target a",
    annotation_position="top left",
)
score_fig.update_layout(
    title="Collective observable over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted cosine score",
)
score_fig.show()

radius_fig = go.Figure()
radius_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=rms_radius,
        mode="lines",
        name="conditioned RMS radius",
    )
)
radius_fig.add_trace(
    go.Scatter(
        x=free_times,
        y=free_rms_radius,
        mode="lines",
        name="free RMS radius",
        opacity=0.8,
    )
)
radius_fig.add_hline(
    y=initial_rms_radius,
    line_dash="dash",
    annotation_text="initial RMS radius",
    annotation_position="top right",
)
radius_fig.update_layout(
    title="Variance-like diagnostic: weighted periodic RMS radius",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted periodic RMS radius",
)
radius_fig.show()

print("initial weighted cosine score:", float(score[0]))
print("final weighted cosine score:", float(score[-1]))
print("target score a:", float(target_score))
print()
print("initial RMS radius:", float(rms_radius[0]))
print("final conditioned RMS radius:", float(rms_radius[-1]))
print("final free RMS radius:", float(free_rms_radius[-1]))
print("conditioned radius reduction factor:", float(rms_radius[-1] / rms_radius[0]))
print("free radius reduction factor:", float(free_rms_radius[-1] / free_rms_radius[0]))



### Seed robustness for the chosen particle count

The next cell reruns the same setup for several seeds and compares the final weighted periodic RMS radius
with the free-diffusion baseline.


In [ ]:

conditioned_final_radii = []
free_final_radii = []

for s in robustness_seeds:
    sim_s = run_conditioned_simulation(s, store_drifts=False)
    conditioned_final_radii.append(
        float(weighted_periodic_rms_radius(sim_s.positions, masses, target_center)[-1])
    )

    _, free_positions_s = simulate_free_diffusion(masses, initial_positions, horizon, step_size, s)
    free_final_radii.append(
        float(weighted_periodic_rms_radius(free_positions_s, masses, target_center)[-1])
    )

conditioned_final_radii = np.array(conditioned_final_radii, dtype=float)
free_final_radii = np.array(free_final_radii, dtype=float)

robust_fig = go.Figure()
robust_fig.add_trace(
    go.Scatter(
        x=robustness_seeds,
        y=conditioned_final_radii,
        mode="lines+markers",
        name="conditioned final RMS radius",
    )
)
robust_fig.add_trace(
    go.Scatter(
        x=robustness_seeds,
        y=free_final_radii,
        mode="lines+markers",
        name="free final RMS radius",
        opacity=0.8,
    )
)
robust_fig.add_hline(
    y=float(conditioned_final_radii.mean()),
    line_dash="dash",
    annotation_text="conditioned mean",
    annotation_position="bottom right",
)
robust_fig.add_hline(
    y=float(free_final_radii.mean()),
    line_dash="dot",
    annotation_text="free mean",
    annotation_position="top right",
)
robust_fig.update_layout(
    title=f"Seed robustness at n = {n_particles}",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="seed",
    yaxis_title="final weighted periodic RMS radius",
)
robust_fig.show()

print("conditioned final RMS radii by seed:")
for s, value in zip(robustness_seeds, conditioned_final_radii):
    print(f"  seed {s}: {value:.6f}")

print()
print("free-diffusion final RMS radii by seed:")
for s, value in zip(robustness_seeds, free_final_radii):
    print(f"  seed {s}: {value:.6f}")

print()
print("conditioned mean final RMS radius:", float(conditioned_final_radii.mean()))
print("conditioned std of final RMS radius:", float(conditioned_final_radii.std()))
print("free mean final RMS radius:", float(free_final_radii.mean()))
print("free std of final RMS radius:", float(free_final_radii.std()))
print("fraction of seeds with conditioned < free:", float(np.mean(conditioned_final_radii < free_final_radii)))



### How far can the particle count go?

This benchmark keeps the same exact Algorithm 2 setup and measures a **single conditioned trajectory**
for a range of particle counts. It is meant as a practical runtime guide for this notebook.

To keep the notebook responsive, the executed version benchmarks up to 32 particles.  
You can extend `benchmark_counts` manually if you want to test larger values.


In [ ]:

benchmark_rows = []

for n in benchmark_counts:
    masses_n = np.ones(n, dtype=float) / n
    initial_positions_n = make_square_perimeter_configuration(
        n,
        center=target_center,
        half_width=square_half_width,
    )
    horizon_n = base_horizon_per_particle / n
    step_size_n = horizon_n / n_steps

    def run_once(seed_value):
        rng = np.random.default_rng(seed_value)
        return simulate_algorithm2_exact(
            masses=masses_n,
            observables=[observable],
            target_vector=np.array([target_score], dtype=float),
            lambda_=lambda_,
            horizon=horizon_n,
            step_size=step_size_n,
            initial_positions=initial_positions_n,
            quadrature_nodes=quadrature_nodes,
            quadrature_weights=quadrature_weights,
            grid_shape=grid_shape,
            rng=rng,
            store_drifts=False,
        )

    t0 = time.perf_counter()
    sim_n = run_once(seed)
    elapsed = time.perf_counter() - t0

    score_n = weighted_observable_score(sim_n.positions, masses_n, observable)
    rms_n = weighted_periodic_rms_radius(sim_n.positions, masses_n, target_center)

    _, free_positions_n = simulate_free_diffusion(masses_n, initial_positions_n, horizon_n, step_size_n, seed)
    free_rms_n = weighted_periodic_rms_radius(free_positions_n, masses_n, target_center)

    benchmark_rows.append(
        {
            "n_particles": n,
            "horizon": horizon_n,
            "runtime_sec": elapsed,
            "initial_rms": float(rms_n[0]),
            "conditioned_final_rms": float(rms_n[-1]),
            "free_final_rms": float(free_rms_n[-1]),
            "conditioned/free_ratio": float(rms_n[-1] / free_rms_n[-1]),
            "final_score": float(score_n[-1]),
        }
    )

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df


In [ ]:

runtime_fig = go.Figure()
runtime_fig.add_trace(
    go.Scatter(
        x=benchmark_df["n_particles"],
        y=benchmark_df["runtime_sec"],
        mode="lines+markers",
        name="conditioned runtime",
    )
)
runtime_fig.update_layout(
    title="Single-run runtime vs particle count",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="number of particles",
    yaxis_title="runtime (seconds)",
)
runtime_fig.show()

ratio_fig = go.Figure()
ratio_fig.add_trace(
    go.Scatter(
        x=benchmark_df["n_particles"],
        y=benchmark_df["conditioned/free_ratio"],
        mode="lines+markers",
        name="final RMS ratio",
    )
)
ratio_fig.add_hline(
    y=1.0,
    line_dash="dash",
    annotation_text="parity with free diffusion",
    annotation_position="top right",
)
ratio_fig.update_layout(
    title="Final RMS radius ratio: conditioned / free",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="number of particles",
    yaxis_title="final RMS ratio",
)
ratio_fig.show()

print("Practical reading:")
print("- counts up to 16 are very comfortable")
print("- 24 to 32 are still fine for a single animation / robustness check")
print("- beyond that, the exact Algorithm 2 solve is still possible but gets noticeably slower")
